# Clinical Case Masking Trial 3

This notebook runs `src/mask_trial3.py`, the same Python program used by `run_mask_trial3.bat`. Trial 3 retains the Trial 2 masking categories while protecting medical concepts and clinically sensitive wording.

The output preserves `PMCID`, `PMID`, and `Case #` and is saved as `clinical_case_mask_trial3_YYYYMMDD_HHMMSS.csv` and `.txt`.

## Required files and Google Drive layout

This notebook does not contain a copy of the masking implementation. It imports `src/mask_trial3.py`, which in turn imports the shared functions in `src/mask_trial2.py`. Both Python files are therefore required. The input CSV is also required at the configured path.

Place the files on Google Drive in the following recommended layout:

```text
MyDrive/
└─ MSc_Project/
   └─ HECTA-UoM-Synthetic4Health/
      ├─ mask-trial3.ipynb
      ├─ src/
      │  ├─ mask_trial2.py
      │  └─ mask_trial3.py
      ├─ example/
      │  └─ karte_example.csv
      └─ mask_trial_outputs/
```

`run_mask_trial3.bat` is only for local Windows/WSL execution and is not required in Google Colab. In Colab, this notebook calls `mask_trial3.main()` directly. Generated files are written to the repository's `mask_trial_outputs/` directory.

The initial configuration processes five cases (`MAX_ROWS = 5`). Change it to `MAX_ROWS = None` in the configuration cell to process every non-empty case.

## 1. Install dependencies

Run this cell first in Google Colab. Restarting the runtime is normally unnecessary.

In [ ]:
%pip install -q pandas spacy
!python -m spacy download en_core_web_sm

## 2. Locate the repository and configure the run

Set `MAX_ROWS = None` to process every non-empty case. Relative input and output paths are resolved from the repository root, matching the batch file.

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys


def mount_drive_if_available() -> None:
    """Mount Google Drive when this notebook runs in Colab."""
    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except ImportError:
        pass
    except Exception as exc:
        print(f"Google Drive mount skipped: {exc}")


mount_drive_if_available()

REPOSITORY_NAME = "HECTA-UoM-Synthetic4Health"
repository_candidates = [
    Path(os.getenv("HECTA_REPO_DIR", "/content/drive/MyDrive/LLMatch2026")),
    Path.cwd(),
    Path.cwd() / REPOSITORY_NAME,
    Path("/content") / REPOSITORY_NAME,
    Path("/content/drive/MyDrive/MSc_Project") / REPOSITORY_NAME,
]
REPO_ROOT = next(
    (path for path in repository_candidates if (path / "src/mask_trial3.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Repository not found. Place this notebook in the repository root or "
        "update repository_candidates."
    )

INPUT_CSV = REPO_ROOT / "example/karte_example.csv"
OUTPUT_DIR = REPO_ROOT / "mask_trial_outputs"
TARGET_COLUMN = "Case presentation EN"
MAX_ROWS: int | None = 5
MASK_RATIO = 0.15
RANDOM_SEED = 42
SPACY_MODEL = "en_core_web_sm"

if not INPUT_CSV.is_file():
    raise FileNotFoundError(f"Input CSV was not found: {INPUT_CSV}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_ROOT / "src"))

print(f"Repository: {REPO_ROOT}")
print(f"Input CSV: {INPUT_CSV}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Max rows: {MAX_ROWS if MAX_ROWS is not None else 'all'}")

## 3. Run the Trial 3 masking program

This calls the same `main()` entry point as `run_mask_trial3.bat`.

In [ ]:
import mask_trial3

arguments = [
    "--input-csv", str(INPUT_CSV),
    "--output-dir", str(OUTPUT_DIR),
    "--target-column", TARGET_COLUMN,
    "--mask-ratio", str(MASK_RATIO),
    "--random-seed", str(RANDOM_SEED),
    "--spacy-model", SPACY_MODEL,
]
if MAX_ROWS is not None:
    arguments.extend(["--max-rows", str(MAX_ROWS)])

exit_code = mask_trial3.main(arguments)
if exit_code != 0:
    raise RuntimeError(f"mask_trial3.py failed with exit code {exit_code}")

## 4. Load and validate the newest output

The checks confirm the Trial 3 version, identifier columns, and absence of protected pathogen names from Mask metadata.

In [ ]:
import json
import re

import pandas as pd

output_files = sorted(
    OUTPUT_DIR.glob("clinical_case_mask_trial3_*.csv"),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
if not output_files:
    raise FileNotFoundError(f"No Trial 3 output CSV was found in {OUTPUT_DIR}")

LATEST_CSV = output_files[0]
masked_df = pd.read_csv(LATEST_CSV, encoding="utf-8")
required_columns = {"PMCID", "PMID", "Case #", "masking_version", "mask_metadata"}
missing_columns = sorted(required_columns - set(masked_df.columns))
if missing_columns:
    raise ValueError(f"Required output columns are missing: {missing_columns}")
if not masked_df["masking_version"].eq("mask-trial3").all():
    raise ValueError("Output contains a masking_version other than mask-trial3")

protected_pathogen_terms = {
    "hanta", "hantaan", "hantavirus", "hantaviral", "hunta",
    "lassa", "lasv", "orthohantavirus", "puumala", "puuv",
}
masked_protected_terms: list[tuple[int, str]] = []
for row_index, raw_metadata in masked_df["mask_metadata"].items():
    metadata = json.loads(raw_metadata)
    for item in metadata:
        original = str(item.get("original_value", "")).lower()
        words = set(re.findall(r"[a-z]+", original))
        matched = sorted(words & protected_pathogen_terms)
        if matched:
            masked_protected_terms.append((int(row_index), str(item.get("original_value", ""))))

if masked_protected_terms:
    raise ValueError(f"Protected pathogen terms were masked: {masked_protected_terms[:10]}")

print(f"Validated CSV: {LATEST_CSV}")
print(f"Rows: {len(masked_df)}")
print("Identifier columns: OK")
print("Protected pathogen Mask check: OK")

## 5. Preview results

In [ ]:
display_columns = [
    "source_row_id", "PMCID", "PMID", "Case #", "disease",
    "masked_token_count", "mask_types", "masked_tokens",
]
display(masked_df[display_columns].head())

if not masked_df.empty:
    preview = masked_df.iloc[0]
    print("Original:")
    print(str(preview["original_text"])[:1500])
    print("\nMasked:")
    print(str(preview["masked_text"])[:1500])
    print("\nProtection policy:")
    print(preview["protection_policy"])